# MT5 AI Trader — Colab Train + Walk-Forward

Notebook này dùng Colab để clone repo, đọc CSV từ Google Drive, train model, chạy walk-forward, rồi lưu `models/` và `reports/` về Drive.

> Colab không chạy MT5 terminal. MT5/paper/demo/live vẫn chạy local Windows.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
%cd /content
!rm -rf mt5-ai-trader
!git clone https://github.com/quoccuongphan300992-cmd/mt5-ai-trader.git
%cd /content/mt5-ai-trader
!git checkout main
!git pull --ff-only origin main
!git log -1 --oneline
!ls -lh


In [ ]:
!python --version
!nproc
!free -h
!df -h


In [ ]:
!pip install -r requirements-colab.txt


## Cấu hình đường dẫn CSV

Upload CSV EURUSD H1 vào Google Drive, ví dụ:

`MyDrive/mt5-ai/data/EURUSD_H1.csv`

Nếu file tên khác, sửa biến `DRIVE_CSV` bên dưới.


In [ ]:
DRIVE_CSV = "/content/drive/MyDrive/mt5-ai/data/EURUSD_H1.csv"
LOCAL_CSV = "data/EURUSD_H1.csv"
SYMBOL = "EURUSD"
TIMEFRAME = "H1"
BARS = 50000


In [ ]:
!ls -lh "/content/drive/MyDrive/mt5-ai/data" || true


In [ ]:
import os, shutil
os.makedirs("data", exist_ok=True)
if not os.path.exists(DRIVE_CSV):
    raise FileNotFoundError(f"CSV not found: {DRIVE_CSV}. Upload CSV to Drive or edit DRIVE_CSV.")
shutil.copy2(DRIVE_CSV, LOCAL_CSV)
print("CSV copied:", LOCAL_CSV, os.path.getsize(LOCAL_CSV), "bytes")


## Auto-Improve Offline Search

Run unattended offline training with config-only search.

Write `models/model.joblib` and `models/metadata.json` only when `candidate_pass=true`.

Do not run MT5, paper, demo, or live trading in Colab.


In [ ]:
# Auto-improve: train only if candidate passes offline validation
!python main.py auto-improve \
  --csv "$LOCAL_CSV" \
  --symbol "$SYMBOL" \
  --timeframe "$TIMEFRAME" \
  --bars "$BARS" \
  --max-rounds 5 \
  --folds 2 \
  --min-trades 60 \
  --min-profit-factor 1.05 \
  --min-expectancy 0.0 \
  --min-positive-fold-ratio 1.0 \
  --max-drawdown-limit 0.20


In [ ]:
# Show result; fail loudly if auto-improve did not complete
import json, os
best_path = "reports/auto_improve_best.json"
if not os.path.exists(best_path):
    raise FileNotFoundError("auto_improve_best.json missing; auto-improve did not complete")
best = json.load(open(best_path, encoding="utf-8"))
print(json.dumps(best, indent=2))
print("candidate_pass:", best.get("candidate_pass"))
print("production_artifacts_written:", best.get("production_artifacts_written"))


In [ ]:
# Save outputs to Google Drive
!mkdir -p "/content/drive/MyDrive/mt5-ai/outputs/models"
!mkdir -p "/content/drive/MyDrive/mt5-ai/outputs/reports"
!cp -r models/* "/content/drive/MyDrive/mt5-ai/outputs/models/" 2>/dev/null || true
!cp -r reports/* "/content/drive/MyDrive/mt5-ai/outputs/reports/" 2>/dev/null || true
!zip -r mt5_ai_outputs.zip models reports
!cp mt5_ai_outputs.zip "/content/drive/MyDrive/mt5-ai/outputs/mt5_ai_outputs.zip"
!find "/content/drive/MyDrive/mt5-ai/outputs" -maxdepth 3 -type f -print
